In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from lightgbm import LGBMClassifier

import dagshub
dagshub.init(repo_owner="AndriaMakharadze", repo_name="IEEE_Fraud_Detection_AM", mlflow=True)

Accessing as AndriaMakharadze

Initialized MLflow to track repo "AndriaMakharadze/IEEE_Fraud_Detection_AM"

Repository AndriaMakharadze/IEEE_Fraud_Detection_AM initialized!

In [2]:
train_transaction = pd.read_csv("../data/train_transaction.csv")
train_identity = pd.read_csv("../data/train_identity.csv")

df = train_transaction.merge(train_identity, on="TransactionID", how="left")
df = df.drop(columns=["TransactionID"], errors="ignore")

y = df["isFraud"]
X = df.drop(columns=["isFraud"])

X = X.sample(150000, random_state=42)
y = y.loc[X.index]

Cleaning

In [3]:
mlflow.set_experiment("LightGBM_Training")

with mlflow.start_run(run_name="LightGBM_Cleaning"):
    null_thresh = 0.8
    cols_to_drop = [c for c in X.columns if X[c].isnull().mean() > null_thresh]
    X = X.drop(columns=cols_to_drop)

    mlflow.log_param("null_threshhold", null_thresh)
    mlflow.log_param("cols_dropped", len(cols_to_drop))
    mlflow.log_metric("cols_remaining", X.shape[1])

    print(f"Dropped {len(cols_to_drop)} high-null columns. Remaining: {X.shape[1]}")

2026/05/04 14:05:06 INFO mlflow.tracking.fluent: Experiment with name 'LightGBM_Training' does not exist. Creating a new experiment.


Dropped 74 high-null columns. Remaining: 358
🏃 View run LightGBM_Cleaning at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/3/runs/68aedf740e964549b3eb26742161b106
🧪 View experiment at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/3


Feature Engineering

In [4]:
with mlflow.start_run(run_name="LightGBM_FeatureEngineering"):
    X["TransactionAmt_log"] = np.log1p(X["TransactionAmt"])
    X["hour_of_day"] = (X["TransactionDT"] / 3600).astype(int) % 24
    X["null_count"] = X.isnull().sum(axis=1)

    new_features = ["TransactionAmt_log", "hour_of_day", "null_count"]
    mlflow.log_param("new_features", new_features)
    mlflow.log_metric("total_cols_after_eng", X.shape[1])
    print("Feature Engineering Done: ", new_features)

Feature Engineering Done:  ['TransactionAmt_log', 'hour_of_day', 'null_count']
🏃 View run LightGBM_FeatureEngineering at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/3/runs/b4209f697872467c8f75c7e7291f6b50
🧪 View experiment at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/3


Feature Selection

In [5]:
with mlflow.start_run(run_name="LightGBM_FeatureSelection"):
    num_only = X.select_dtypes(include=["int64", "float64"]).fillna(0)
    corr_matrix = num_only.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    drop_corr = [col for col in upper.columns if any(upper[col] > 0.95)]
    X = X.drop(columns=drop_corr, errors="ignore")

    mlflow.log_param("corr_threshold", 0.95)
    mlflow.log_metric("cols_dropped_corr", len(drop_corr))
    mlflow.log_metric("cols_remaining", X.shape[1])
    print(f"Dropped {len(drop_corr)} correlated columns. Remaining: {X.shape[1]}")

Dropped 109 correlated columns. Remaining: 252
🏃 View run LightGBM_FeatureSelection at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/3/runs/dfcfaced0a6e41bd98c120442532dc04
🧪 View experiment at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/3


In [6]:
num_cols = X.select_dtypes(include=["int64", "float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

num_pipeline = Pipeline([("imputer", SimpleImputer(strategy="median"))])

cat_pipeline = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))])

preprocessor = ColumnTransformer([("num", num_pipeline, num_cols), ("cat", cat_pipeline, cat_cols)])

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

Training

In [7]:
mlflow.set_experiment("LightGBM_Training")

with mlflow.start_run(run_name="LightGBM_Training"):
    pipeline = Pipeline([
        ("preprocessing", preprocessor),
        ("model", LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            random_state=42,
            n_jobs=-1
        ))
    ])

    pipeline.fit(X_train, y_train)

    preds = pipeline.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, preds)

    mlflow.log_param("n_estimators", 300)
    mlflow.log_param("learning_rate", 0.05)
    mlflow.log_param("max_depth", 6)
    mlflow.log_metric("auc", auc)

    mlflow.sklearn.log_model(pipeline, "pipeline_model", registered_model_name="LightGBM_FraudDetection")

    print("AUC:", auc)

[LightGBM] [Info] Number of positive: 4353, number of negative: 115647
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.018129 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 18634
[LightGBM] [Info] Number of data points in the train set: 120000, number of used features: 249
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.036275 -> initscore=-3.279677
[LightGBM] [Info] Start training from score -3.279677
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

c:\Users\andriam\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
2026/05/04 14:05:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/04 14:05:44 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'LightGBM_FraudDetection'.
2026/05/04 14:05:53 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: LightGBM_FraudDetection, version 1
Created version '1' of model '

AUC: 0.9164968780467322
🏃 View run LightGBM_Training at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/3/runs/b1a4e2a41c22427aac07cab16f2d738d
🧪 View experiment at: https://dagshub.com/AndriaMakharadze/IEEE_Fraud_Detection_AM.mlflow/#/experiments/3
